# Model Experiment: ARIMA

This notebook evaluates ARIMA as a classical univariate time-series baseline for weekly Walmart Store-Dept sales forecasting.

Unlike neural models ARIMA does not learn shared representations across all Store-Dept pairs and does not directly use static or future covariates. Each Store-Dept series is modeled mainly from its own historical `Weekly_Sales` values.

The goal of this notebook is not exhaustive ARIMA tuning. Focus is to explain the classical forecasting theory, apply ARIMA in a scalable way, evaluate it with the same WMAE metric and compare its limitations against feature-aware models such as LightGBM, TFT, PatchTST-X and so on.

ARIMA stands for **AutoRegressive Integrated Moving Average** and is written as:

`ARIMA(p, d, q)`

where:

- `p` is the number of autoregressive lags
- `d` is the number of differencing operations
- `q` is the number of lagged residual errors used by the moving-average part

The model first differences the original series `d` times to make it more stationary. It then fits a linear model on the transformed series using previous transformed values and previous forecast errors.

For transformed series `z[t]`, the general idea is:

`z[t] ≈ c + φ1 z[t-1] + ... + φp z[t-p] + θ1 e[t-1] + ... + θq e[t-q] + e[t]`

Here:

- `φ` coefficients measure the effect of previous values
- `θ` coefficients measure the effect of previous residual errors
- `e[t]` is the current unexplained shock or residual

Unlike neural networks ARIMA parameters are usually estimated through statistical optimization, commonly maximum likelihood estimation, rather than minibatch gradient descent.

The ARIMA experiment uses each Store-Dept sales history as separate univariate time series.

For our task:

- target series: weekly `Weekly_Sales`
- forecast horizon: next 39 weeks
- main validation split: last-39 weeks
- secondary validation split: calendar-aligned validation, depending on runtime
- evaluation metric: Weighted Mean Absolute Error (WMAE)

Because the dataset contains thousands of short and heterogeneous Store-Dept series, we avoid heavy per-series hyperparameter search. Instead, we use a small set of simple ARIMA orders for demonstration and fixed practical ARIMA configuration for scalable validation.

ARIMA is expected to be useful as classical baseline, but it has important limitations for this dataset:

- it models each Store-Dept series mostly independently
- it does not naturally share information across stores or departments
- it does not directly use future holiday, markdown, store type, or economic covariates
- many Store-Dept series are short, sparse, noisy or irregular

For failed or unsuitable ARIMA fits, we use a simple fallback forecast based on recent historical sales.

## Setup and Imports

In [1]:
from pathlib import Path
import sys
import os
import json
import random
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tools.sm_exceptions import ConvergenceWarning

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [2]:
# cwd = Path.cwd().resolve()
# repo_root = cwd if (cwd / "src").exists() else cwd.parent

# if str(repo_root) not in sys.path:
#     sys.path.insert(0, str(repo_root))

# ==========================================

# repo_root = Path("/content/drive/MyDrive/Machine Learning/Walmart").resolve()

# assert (repo_root / "src").exists(), f"src not found at {repo_root / 'src'}"

# print("Repo root:", repo_root)
# print("src exists:", (repo_root / "src").exists())
# print("data exists:", (repo_root / "data" / "raw").exists())

# os.chdir(repo_root)

# ==========================================

import shutil
import zipfile

KAGGLE_INPUT = Path("/kaggle/input")
repo_root = Path("/kaggle/working/Walmart").resolve()
data_raw_dir = repo_root / "data" / "raw"

repo_root.mkdir(parents=True, exist_ok=True)
data_raw_dir.mkdir(parents=True, exist_ok=True)

print("Available /kaggle/input folders:")
for p in KAGGLE_INPUT.iterdir():
    print(" -", p)


src_candidates = [
    p for p in KAGGLE_INPUT.rglob("src")
    if p.is_dir() and (p / "data").exists() and (p / "features").exists()
]

if not src_candidates:
    raise FileNotFoundError(
        "Could not find your project src/ folder under /kaggle/input. "
        "Make sure your Kaggle Dataset contains the src directory."
    )

source_src = src_candidates[0]
target_src = repo_root / "src"

if target_src.exists():
    shutil.rmtree(target_src)

shutil.copytree(source_src, target_src)

print("\nCopied src from:", source_src)
print("Copied src to:", target_src)

required_csvs = ["train.csv", "test.csv", "features.csv", "stores.csv"]

for csv_name in required_csvs:
    matches = list(KAGGLE_INPUT.rglob(csv_name))
    if matches:
        shutil.copy2(matches[0], data_raw_dir / csv_name)
        print(f"Copied {csv_name} from:", matches[0])

for zip_path in KAGGLE_INPUT.rglob("*.zip"):
    try:
        with zipfile.ZipFile(zip_path, "r") as z:
            names = z.namelist()
            wanted = [name for name in names if Path(name).name in required_csvs]

            for name in wanted:
                out_name = Path(name).name
                with z.open(name) as src_file, open(data_raw_dir / out_name, "wb") as dst_file:
                    shutil.copyfileobj(src_file, dst_file)

                print(f"Extracted {out_name} from:", zip_path)
    except zipfile.BadZipFile:
        pass

missing = [name for name in required_csvs if not (data_raw_dir / name).exists()]

if missing:
    print("\nFiles currently in data/raw:")
    for p in data_raw_dir.iterdir():
        print(" -", p.name)

    raise FileNotFoundError(f"Missing required raw files: {missing}")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

os.chdir(repo_root)

print("\nRepo root:", repo_root)
print("src exists:", (repo_root / "src").exists())
print("data exists:", data_raw_dir.exists())
print("Raw data files:", sorted(p.name for p in data_raw_dir.iterdir()))

from src.data import load_raw_data, last_n_weeks_split, calendar_aligned_split

Available /kaggle/input folders:
 - /kaggle/input/competitions
 - /kaggle/input/datasets

Copied src from: /kaggle/input/datasets/myvari/walmart-project-code/src
Copied src to: /kaggle/working/Walmart/src
Copied stores.csv from: /kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/stores.csv
Extracted train.csv from: /kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/train.csv.zip
Extracted features.csv from: /kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/features.csv.zip
Extracted test.csv from: /kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/test.csv.zip

Repo root: /kaggle/working/Walmart
src exists: True
data exists: True
Raw data files: ['features.csv', 'stores.csv', 'test.csv', 'train.csv']


In [3]:
DATA_DIR = repo_root / "data" / "raw"

PREDICTION_LENGTH = 39
SEED = 42

MLFLOW_EXPERIMENT_NAME = "ARIMA_Training"

PRIMARY_ARIMA_ORDER = (1, 1, 1)

ARIMA_CANDIDATE_ORDERS = [
    (0, 1, 0),
    (1, 0, 0),
    (1, 1, 0),
    (0, 1, 1),
    (1, 1, 1),
    (2, 1, 0),
]

MIN_OBSERVATIONS_FOR_ARIMA = 30
FALLBACK_METHOD = "last_13_mean"

print("Repo root:", repo_root)
print("Data dir:", DATA_DIR)
print("Primary ARIMA order:", PRIMARY_ARIMA_ORDER)
print("Candidate ARIMA orders:", ARIMA_CANDIDATE_ORDERS)

Repo root: /kaggle/working/Walmart
Data dir: /kaggle/working/Walmart/data/raw
Primary ARIMA order: (1, 1, 1)
Candidate ARIMA orders: [(0, 1, 0), (1, 0, 0), (1, 1, 0), (0, 1, 1), (1, 1, 1), (2, 1, 0)]


In [5]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)

set_seed(SEED)

## Dagshub/Mlflow initialization

In [6]:
pip install dagshub mlflow pandas matplotlib seaborn skops --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.3/273.3 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 83.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 94.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.3/121.3 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [8]:
import dagshub
import mlflow

dagshub.init(repo_owner='LukaBatilashvili07', repo_name='walmart-sales-forecasting', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=fbf7cfce-6fe3-4cef-8d0f-d3e46f1916ea&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=8d3bc751c29526a0b48bb4959bd9dfc6736b5391ea69f191b220250b840f801d




Accessing as myvari

Initialized MLflow to track repo "LukaBatilashvili07/walmart-sales-forecasting"

Repository LukaBatilashvili07/walmart-sales-forecasting initialized!

## Load Dataset and Time Split

In [9]:
train, test, stores, features = load_raw_data(DATA_DIR)

for df in [train, test, features]:
    df["Date"] = pd.to_datetime(df["Date"])

train = train.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
test = test.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
features = features.sort_values(["Store", "Date"]).reset_index(drop=True)
stores = stores.sort_values("Store").reset_index(drop=True)

print("train:", train.shape)
print("test:", test.shape)
print("stores:", stores.shape)
print("features:", features.shape)

print("\nTrain date range:")
print(train["Date"].min(), "->", train["Date"].max())

print("\nTest date range:")
print(test["Date"].min(), "->", test["Date"].max())

print("\nUnique train dates:", train["Date"].nunique())
print("Unique test dates:", test["Date"].nunique())

print("\nTrain Store-Dept pairs:", train[["Store", "Dept"]].drop_duplicates().shape[0])
print("Test Store-Dept pairs:", test[["Store", "Dept"]].drop_duplicates().shape[0])

display(train.head())
display(test.head())
display(stores.head())
display(features.head())

train: (421570, 5)
test: (115064, 4)
stores: (45, 3)
features: (8190, 12)

Train date range:
2010-02-05 00:00:00 -> 2012-10-26 00:00:00

Test date range:
2012-11-02 00:00:00 -> 2013-07-26 00:00:00

Unique train dates: 143
Unique test dates: 39

Train Store-Dept pairs: 3331
Test Store-Dept pairs: 3169


,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,1,2010-02-12,46039.49,True
2,1,1,2010-02-19,41595.55,False
3,1,1,2010-02-26,19403.54,False
4,1,1,2010-03-05,21827.90,False


,Store,Dept,Date,IsHoliday
0,1,1,2012-11-02,False
1,1,1,2012-11-09,False
2,1,1,2012-11-16,False
3,1,1,2012-11-23,True
4,1,1,2012-11-30,False


,Store,Type,Size
0,1,A,151315
1,2,A,202307
2,3,B,37392
3,4,A,205863
4,5,B,34875


,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,1,2010-02-05,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False
1,1,2010-02-12,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True
2,1,2010-02-19,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False
3,1,2010-02-26,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,False
4,1,2010-03-05,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,False


In [10]:
assert {"Store", "Dept", "Date", "Weekly_Sales", "IsHoliday"}.issubset(train.columns)
assert {"Store", "Dept", "Date", "IsHoliday"}.issubset(test.columns)
assert {"Store", "Type", "Size"}.issubset(stores.columns)
assert {"Store", "Date", "IsHoliday"}.issubset(features.columns)

assert train["Date"].notna().all()
assert test["Date"].notna().all()
assert train["Weekly_Sales"].notna().all()

assert pd.api.types.is_numeric_dtype(train["Weekly_Sales"])

train_date_deltas = np.diff(sorted(train["Date"].unique())).astype("timedelta64[D]").astype(int)
test_date_deltas = np.diff(sorted(test["Date"].unique())).astype("timedelta64[D]").astype(int)

assert np.all(train_date_deltas == 7)
assert np.all(test_date_deltas == 7)

print("Raw data checks passed.")

Raw data checks passed.


In [11]:
# split A: last 39 weeks of train

last39_train_raw, last39_valid_raw = last_n_weeks_split(
    train,
    n_weeks=PREDICTION_LENGTH,
    date_col="Date",
)

last39_train_raw = last39_train_raw.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
last39_valid_raw = last39_valid_raw.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)

print("Last-39 split")
print("last39_train_raw:", last39_train_raw.shape)
print("last39_valid_raw:", last39_valid_raw.shape)

print("\nTrain split date range:")
print(last39_train_raw["Date"].min(), "->", last39_train_raw["Date"].max())

print("\nValidation split date range:")
print(last39_valid_raw["Date"].min(), "->", last39_valid_raw["Date"].max())

print("\nUnique train dates:", last39_train_raw["Date"].nunique())
print("Unique validation dates:", last39_valid_raw["Date"].nunique())

print("\nTrain Store-Dept pairs:", last39_train_raw[["Store", "Dept"]].drop_duplicates().shape[0])
print("Validation Store-Dept pairs:", last39_valid_raw[["Store", "Dept"]].drop_duplicates().shape[0])

assert last39_train_raw["Date"].max() < last39_valid_raw["Date"].min()
assert last39_valid_raw["Date"].nunique() == PREDICTION_LENGTH

print("Last-39 split checks passed.")

Last-39 split
last39_train_raw: (305982, 5)
last39_valid_raw: (115588, 5)

Train split date range:
2010-02-05 00:00:00 -> 2012-01-27 00:00:00

Validation split date range:
2012-02-03 00:00:00 -> 2012-10-26 00:00:00

Unique train dates: 104
Unique validation dates: 39

Train Store-Dept pairs: 3306
Validation Store-Dept pairs: 3204
Last-39 split checks passed.


In [12]:
# split B: calendar-aligned validation

calendar_train_raw, calendar_valid_raw = calendar_aligned_split(
    train,
    valid_start="2011-11-04",
    valid_end="2012-07-27",
    date_col="Date",
)

calendar_train_raw = calendar_train_raw.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
calendar_valid_raw = calendar_valid_raw.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)

print("Calendar-aligned split")
print("calendar_train_raw:", calendar_train_raw.shape)
print("calendar_valid_raw:", calendar_valid_raw.shape)

print("\nCalendar train date range:")
print(calendar_train_raw["Date"].min(), "->", calendar_train_raw["Date"].max())

print("\nCalendar validation date range:")
print(calendar_valid_raw["Date"].min(), "->", calendar_valid_raw["Date"].max())

print("\nUnique calendar train dates:", calendar_train_raw["Date"].nunique())
print("Unique calendar validation dates:", calendar_valid_raw["Date"].nunique())

print("\nCalendar train Store-Dept pairs:", calendar_train_raw[["Store", "Dept"]].drop_duplicates().shape[0])
print("Calendar validation Store-Dept pairs:", calendar_valid_raw[["Store", "Dept"]].drop_duplicates().shape[0])

assert calendar_train_raw["Date"].max() < calendar_valid_raw["Date"].min()
assert calendar_valid_raw["Date"].nunique() == PREDICTION_LENGTH

print("Calendar-aligned split checks passed.")

Calendar-aligned split
calendar_train_raw: (267184, 5)
calendar_valid_raw: (115856, 5)

Calendar train date range:
2010-02-05 00:00:00 -> 2011-10-28 00:00:00

Calendar validation date range:
2011-11-04 00:00:00 -> 2012-07-27 00:00:00

Unique calendar train dates: 91
Unique calendar validation dates: 39

Calendar train Store-Dept pairs: 3254
Calendar validation Store-Dept pairs: 3233
Calendar-aligned split checks passed.


In [14]:
def summarize_validation_coverage(train_part, valid_part, split_name):
    train_pairs = train_part[["Store", "Dept"]].drop_duplicates()
    valid_pairs = valid_part[["Store", "Dept"]].drop_duplicates()

    overlap_pairs = valid_pairs.merge(
        train_pairs,
        on=["Store", "Dept"],
        how="inner",
    )

    valid_group_sizes = (
        valid_part.groupby(["Store", "Dept"])
        .size()
        .reset_index(name="valid_weeks")
    )

    full_horizon_pairs = valid_group_sizes[
        valid_group_sizes["valid_weeks"] == PREDICTION_LENGTH
    ]

    print("=" * 80)
    print(split_name)
    print("Train pairs:", len(train_pairs))
    print("Validation pairs:", len(valid_pairs))
    print("Validation pairs also present in train:", len(overlap_pairs))
    print("Validation rows evaluated by ARIMA:", len(valid_part))
    print("Full 39-week validation pairs:", len(full_horizon_pairs))
    print("Full 39-week validation rows:", len(full_horizon_pairs) * PREDICTION_LENGTH)

    print("\nValidation weeks per Store-Dept:")
    display(valid_group_sizes["valid_weeks"].describe())

    return valid_group_sizes


last39_valid_group_sizes = summarize_validation_coverage(
    last39_train_raw,
    last39_valid_raw,
    "Last-39 validation coverage",
)

calendar_valid_group_sizes = summarize_validation_coverage(
    calendar_train_raw,
    calendar_valid_raw,
    "Calendar-aligned validation coverage",
)

Last-39 validation coverage
Train pairs: 3306
Validation pairs: 3204
Validation pairs also present in train: 3179
Validation rows evaluated by ARIMA: 115588
Full 39-week validation pairs: 2762
Full 39-week validation rows: 107718

Validation weeks per Store-Dept:


count    3204.000000
mean       36.076155
std         8.879486
min         1.000000
25%        39.000000
50%        39.000000
75%        39.000000
max        39.000000
Name: valid_weeks, dtype: float64

Calendar-aligned validation coverage
Train pairs: 3254
Validation pairs: 3233
Validation pairs also present in train: 3165
Validation rows evaluated by ARIMA: 115856
Full 39-week validation pairs: 2762
Full 39-week validation rows: 107718

Validation weeks per Store-Dept:


count    3233.000000
mean       35.835447
std         9.229179
min         1.000000
25%        39.000000
50%        39.000000
75%        39.000000
max        39.000000
Name: valid_weeks, dtype: float64

## ARIMA Series Preparation and Fallback Strategy

In [15]:
arima_required_train_cols = ["Store", "Dept", "Date", "Weekly_Sales", "IsHoliday"]
arima_required_future_cols = ["Store", "Dept", "Date", "IsHoliday"]

for name, df in {
    "last39_train_raw": last39_train_raw,
    "last39_valid_raw": last39_valid_raw,
    "calendar_train_raw": calendar_train_raw,
    "calendar_valid_raw": calendar_valid_raw,
}.items():
    required = arima_required_train_cols if "train" in name else arima_required_future_cols
    missing = [col for col in required if col not in df.columns]
    assert not missing, f"{name} is missing columns: {missing}"

    assert df["Date"].notna().all(), f"{name} has missing Date values"
    assert df[["Store", "Dept"]].notna().all().all(), f"{name} has missing Store/Dept values"

    if "Weekly_Sales" in df.columns:
        assert df["Weekly_Sales"].notna().all(), f"{name} has missing Weekly_Sales values"

print("ARIMA raw input checks passed.")

ARIMA raw input checks passed.


In [16]:
def summarize_pair_overlap(history_df, future_df, split_name):
    history_pairs = (
        history_df[["Store", "Dept"]]
        .drop_duplicates()
        .sort_values(["Store", "Dept"])
        .reset_index(drop=True)
    )

    future_pairs = (
        future_df[["Store", "Dept"]]
        .drop_duplicates()
        .sort_values(["Store", "Dept"])
        .reset_index(drop=True)
    )

    overlap_pairs = future_pairs.merge(
        history_pairs,
        on=["Store", "Dept"],
        how="inner",
    )

    unseen_pairs = future_pairs.merge(
        history_pairs,
        on=["Store", "Dept"],
        how="left",
        indicator=True,
    )
    unseen_pairs = unseen_pairs[unseen_pairs["_merge"] == "left_only"].drop(columns="_merge")

    print("=" * 80)
    print(split_name)
    print("History pairs:", len(history_pairs))
    print("Future/validation pairs:", len(future_pairs))
    print("Overlapping pairs:", len(overlap_pairs))
    print("Unseen future pairs:", len(unseen_pairs))
    print("Future/validation rows:", len(future_df))

    if len(unseen_pairs) > 0:
        print("\nUnseen pairs:")
        display(unseen_pairs.head(20))

    return {
        "history_pairs": history_pairs,
        "future_pairs": future_pairs,
        "overlap_pairs": overlap_pairs,
        "unseen_pairs": unseen_pairs,
    }


last39_pair_coverage = summarize_pair_overlap(
    last39_train_raw,
    last39_valid_raw,
    "Last-39 validation pair coverage",
)

calendar_pair_coverage = summarize_pair_overlap(
    calendar_train_raw,
    calendar_valid_raw,
    "Calendar-aligned validation pair coverage",
)

test_pair_coverage = summarize_pair_overlap(
    train,
    test,
    "Final test pair coverage",
)

Last-39 validation pair coverage
History pairs: 3306
Future/validation pairs: 3204
Overlapping pairs: 3179
Unseen future pairs: 25
Future/validation rows: 115588

Unseen pairs:


,Store,Dept
206,3,83
508,7,99
875,12,99
1099,15,99
1173,16,99
1247,17,99
1358,19,39
1515,21,50
1542,21,99
1689,23,99


Calendar-aligned validation pair coverage
History pairs: 3254
Future/validation pairs: 3233
Overlapping pairs: 3165
Unseen future pairs: 68
Future/validation rows: 115856

Unseen pairs:


,Store,Dept
57,1,77
132,2,77
209,3,83
278,4,77
351,5,77
422,6,77
498,7,77
515,7,99
573,8,77
588,8,96


Final test pair coverage
History pairs: 3331
Future/validation pairs: 3169
Overlapping pairs: 3158
Unseen future pairs: 11
Future/validation rows: 115064

Unseen pairs:


,Store,Dept
360,5,99
648,9,99
723,10,99
1275,18,43
1714,24,43
1821,25,99
2413,34,39
2545,36,30
2605,37,29
2948,42,30


In [21]:
def get_store_dept_history(df, store, dept):
    history = (
        df[(df["Store"] == store) & (df["Dept"] == dept)]
        .copy()
        .sort_values("Date")
        .reset_index(drop=True)
    )
    return history


def make_observed_sales_series(history_df):
    if history_df.empty:
        return pd.Series(dtype=float)

    history_df = history_df.copy()
    history_df["Date"] = pd.to_datetime(history_df["Date"])

    y = (
        history_df
        .groupby("Date")["Weekly_Sales"]
        .sum()
        .sort_index()
        .astype(float)
    )

    y = y.replace([np.inf, -np.inf], np.nan).dropna()

    return y


def prepare_arima_input(y):
    y = pd.Series(np.asarray(y, dtype=float))
    y = y.replace([np.inf, -np.inf], np.nan).dropna()
    return y

In [22]:
def build_fallback_stats(history_df, lookback_weeks=13):
    history_df = history_df.copy()
    history_df["Date"] = pd.to_datetime(history_df["Date"])

    recent_dates = sorted(history_df["Date"].unique())[-lookback_weeks:]
    recent_df = history_df[history_df["Date"].isin(recent_dates)].copy()

    pair_recent_mean = (
        recent_df.groupby(["Store", "Dept"])["Weekly_Sales"]
        .mean()
        .to_dict()
    )

    dept_recent_mean = (
        recent_df.groupby("Dept")["Weekly_Sales"]
        .mean()
        .to_dict()
    )

    store_recent_mean = (
        recent_df.groupby("Store")["Weekly_Sales"]
        .mean()
        .to_dict()
    )

    global_recent_mean = float(recent_df["Weekly_Sales"].mean())

    return {
        "lookback_weeks": lookback_weeks,
        "pair_recent_mean": pair_recent_mean,
        "dept_recent_mean": dept_recent_mean,
        "store_recent_mean": store_recent_mean,
        "global_recent_mean": global_recent_mean,
    }


def get_fallback_value(store, dept, fallback_stats):
    pair_key = (store, dept)

    if pair_key in fallback_stats["pair_recent_mean"]:
        value = fallback_stats["pair_recent_mean"][pair_key]
        source = "pair_recent_mean"
    elif dept in fallback_stats["dept_recent_mean"]:
        value = fallback_stats["dept_recent_mean"][dept]
        source = "dept_recent_mean"
    elif store in fallback_stats["store_recent_mean"]:
        value = fallback_stats["store_recent_mean"][store]
        source = "store_recent_mean"
    else:
        value = fallback_stats["global_recent_mean"]
        source = "global_recent_mean"

    value = max(float(value), 0.0)
    return value, source


def fallback_forecast(store, dept, horizon, fallback_stats):
    value, source = get_fallback_value(store, dept, fallback_stats)
    forecast = np.full(horizon, value, dtype=float)
    return forecast, source

In [24]:
last39_fallback_stats = build_fallback_stats(
    last39_train_raw,
    lookback_weeks=13,
)

calendar_fallback_stats = build_fallback_stats(
    calendar_train_raw,
    lookback_weeks=13,
)

final_fallback_stats = build_fallback_stats(
    train,
    lookback_weeks=13,
)

print("Last-39 global fallback:", last39_fallback_stats["global_recent_mean"])
print("Calendar global fallback:", calendar_fallback_stats["global_recent_mean"])
print("Final train global fallback:", final_fallback_stats["global_recent_mean"])

print("\nFallback stats built.")

Last-39 global fallback: 17195.08554590443
Calendar global fallback: 15456.849575444638
Final train global fallback: 15620.503259018944

Fallback stats built.


In [25]:
def is_arima_eligible(history_df, split_end_date, min_observations=30, max_weeks_since_last_observation=13):
    if history_df.empty:
        return False, "no_pair_history"

    history_df = history_df.copy()
    history_df["Date"] = pd.to_datetime(history_df["Date"])

    n_observations = len(history_df)
    if n_observations < min_observations:
        return False, "too_few_observations"

    last_observed_date = history_df["Date"].max()
    split_end_date = pd.to_datetime(split_end_date)

    weeks_since_last_observation = (split_end_date - last_observed_date).days / 7

    if weeks_since_last_observation > max_weeks_since_last_observation:
        return False, "stale_series"

    return True, "eligible"

In [26]:
unseen_test_pairs = test_pair_coverage["unseen_pairs"]

print("Unseen final test pairs:", len(unseen_test_pairs))

if len(unseen_test_pairs) > 0:
    fallback_examples = []

    for _, row in unseen_test_pairs.head(20).iterrows():
        store = row["Store"]
        dept = row["Dept"]

        value, source = get_fallback_value(
            store=store,
            dept=dept,
            fallback_stats=final_fallback_stats,
        )

        fallback_examples.append({
            "Store": store,
            "Dept": dept,
            "fallback_value": value,
            "fallback_source": source,
        })

    display(pd.DataFrame(fallback_examples))

Unseen final test pairs: 11


,Store,Dept,fallback_value,fallback_source
0,5,99,102.417850,dept_recent_mean
1,9,99,102.417850,dept_recent_mean
2,10,99,102.417850,dept_recent_mean
3,18,43,15145.121692,store_recent_mean
4,24,43,18925.655091,store_recent_mean
5,25,99,102.417850,dept_recent_mean
6,34,39,15.231111,dept_recent_mean
7,36,30,3638.152973,dept_recent_mean
8,37,29,5290.346928,dept_recent_mean
9,42,30,3638.152973,dept_recent_mean


## ARIMA Preprocessing logging

In [34]:
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run(run_name="ARIMA_Data_Preparation") as run:
    mlflow.log_param("model_family", "ARIMA")
    mlflow.log_param("preprocessing_type", "raw_univariate_series")
    mlflow.log_param("base_preprocessor", "not_used")
    mlflow.log_param("neural_preprocessor", "not_used")
    mlflow.log_param("target_scaling", "not_used")
    mlflow.log_param("missing_weeks_strategy", "observed_points_only_no_zero_fill")
    mlflow.log_param("fallback_strategy", "hierarchical_recent_mean")
    mlflow.log_param("fallback_lookback_weeks", 13)
    mlflow.log_param("primary_validation_strategy", "last_39_weeks")
    mlflow.log_param("secondary_validation_strategy", "calendar_aligned_39_weeks")

    mlflow.log_metric("raw_train_rows", len(train))
    mlflow.log_metric("raw_test_rows", len(test))
    mlflow.log_metric("raw_store_rows", len(stores))
    mlflow.log_metric("raw_feature_rows", len(features))

    mlflow.log_metric("last39_train_rows", len(last39_train_raw))
    mlflow.log_metric("last39_valid_rows", len(last39_valid_raw))
    mlflow.log_metric("calendar_train_rows", len(calendar_train_raw))
    mlflow.log_metric("calendar_valid_rows", len(calendar_valid_raw))

    mlflow.log_metric("last39_history_pairs", len(last39_pair_coverage["history_pairs"]))
    mlflow.log_metric("last39_future_pairs", len(last39_pair_coverage["future_pairs"]))
    mlflow.log_metric("last39_unseen_future_pairs", len(last39_pair_coverage["unseen_pairs"]))

    mlflow.log_metric("calendar_history_pairs", len(calendar_pair_coverage["history_pairs"]))
    mlflow.log_metric("calendar_future_pairs", len(calendar_pair_coverage["future_pairs"]))
    mlflow.log_metric("calendar_unseen_future_pairs", len(calendar_pair_coverage["unseen_pairs"]))

    mlflow.log_metric("test_history_pairs", len(test_pair_coverage["history_pairs"]))
    mlflow.log_metric("test_future_pairs", len(test_pair_coverage["future_pairs"]))
    mlflow.log_metric("test_unseen_future_pairs", len(test_pair_coverage["unseen_pairs"]))

    mlflow.log_metric("last39_global_fallback_mean", last39_fallback_stats["global_recent_mean"])
    mlflow.log_metric("calendar_global_fallback_mean", calendar_fallback_stats["global_recent_mean"])
    mlflow.log_metric("final_global_fallback_mean", final_fallback_stats["global_recent_mean"])

    preprocessing_config = {
        "model_family": "ARIMA",
        "preprocessing_type": "raw_univariate_series",
        "missing_weeks_strategy": "observed_points_only_no_zero_fill",
        "fallback_strategy": "hierarchical_recent_mean",
        "fallback_lookback_weeks": 13,
        "primary_validation_strategy": "last_39_weeks",
        "secondary_validation_strategy": "calendar_aligned_39_weeks",
        "prediction_length": PREDICTION_LENGTH,
        "min_observations_for_arima": MIN_OBSERVATIONS_FOR_ARIMA,
        "fallback_method": FALLBACK_METHOD,
        "last39_unseen_future_pairs": int(len(last39_pair_coverage["unseen_pairs"])),
        "calendar_unseen_future_pairs": int(len(calendar_pair_coverage["unseen_pairs"])),
        "test_unseen_future_pairs": int(len(test_pair_coverage["unseen_pairs"])),
    }

    config_path = repo_root / "arima_preprocessing_config.json"

    with open(config_path, "w") as f:
        json.dump(preprocessing_config, f, indent=2)

    mlflow.log_artifact(str(config_path), artifact_path="config")
    config_path.unlink()

    arima_preprocessing_run_id = run.info.run_id

print("Logged ARIMA preprocessing run:", arima_preprocessing_run_id)

2026/07/25 16:54:21 INFO mlflow.tracking.fluent: Experiment with name 'ARIMA_Training' does not exist. Creating a new experiment.


🏃 View run ARIMA_Data_Preparation at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/9/runs/b9ce5495b41e49fa9085c8df071771ff
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/9
Logged ARIMA preprocessing run: b9ce5495b41e49fa9085c8df071771ff


## Evaluation utilities

In [37]:
def weighted_mae_np(y_true, y_pred, is_holiday):
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=np.float64).reshape(-1)
    is_holiday = np.asarray(is_holiday).reshape(-1).astype(bool)

    weights = np.where(is_holiday, 5.0, 1.0)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)


def evaluate_arima_validation(valid_df: pd.DataFrame, pred_rows: pd.DataFrame):
    valid_df = valid_df.copy()
    pred_rows = pred_rows.copy()

    valid_df["Date"] = pd.to_datetime(valid_df["Date"])
    pred_rows["Date"] = pd.to_datetime(pred_rows["Date"])

    eval_df = valid_df.merge(
        pred_rows,
        on=["Store", "Dept", "Date"],
        how="left",
    )

    assert len(eval_df) == len(valid_df)
    assert eval_df["Weekly_Sales_pred"].notna().all()

    valid_wmae = weighted_mae_np(
        eval_df["Weekly_Sales"],
        eval_df["Weekly_Sales_pred"],
        eval_df["IsHoliday"],
    )

    valid_mae = np.mean(
        np.abs(eval_df["Weekly_Sales"].values - eval_df["Weekly_Sales_pred"].values)
    )

    return {
        "valid_wmae": valid_wmae,
        "valid_mae": valid_mae,
        "eval_df": eval_df,
    }

## ARIMA Model Definitions

In [28]:
def safe_arima_forecast(y_train, horizon: int, order: tuple[int, int, int]):
    """
    Fit ARIMA on one Store-Dept sales sequence and return non-negative forecast.

    Returns None if fitting or forecasting fails.
    """
    y_train = prepare_arima_input(y_train)

    if len(y_train) == 0:
        return None

    try:
        model = ARIMA(
            y_train,
            order=order,
            enforce_stationarity=False,
            enforce_invertibility=False,
        )

        fitted = model.fit()
        forecast = fitted.forecast(steps=horizon)
        forecast = np.asarray(forecast, dtype=float)

        if len(forecast) != horizon:
            return None

        if not np.all(np.isfinite(forecast)):
            return None

        return np.clip(forecast, 0.0, None)

    except Exception:
        return None

## Training utilities

In [29]:
def forecast_one_pair_arima(
    history_df: pd.DataFrame,
    future_dates,
    store: int,
    dept: int,
    order: tuple[int, int, int],
    fallback_stats: dict,
    split_end_date,
    min_observations: int = MIN_OBSERVATIONS_FOR_ARIMA,
    max_weeks_since_last_observation: int = 13,
):
    """
    Forecast one Store-Dept pair for the full future date horizon.
    """
    future_dates = pd.to_datetime(pd.Index(future_dates))
    horizon = len(future_dates)

    pair_history = get_store_dept_history(history_df, store, dept)

    eligible, reason = is_arima_eligible(
        pair_history,
        split_end_date=split_end_date,
        min_observations=min_observations,
        max_weeks_since_last_observation=max_weeks_since_last_observation,
    )

    if eligible:
        y_train = make_observed_sales_series(pair_history)
        forecast = safe_arima_forecast(y_train, horizon=horizon, order=order)

        if forecast is not None:
            method = "arima"
            fallback_source = None
            final_reason = "arima_success"
        else:
            forecast, fallback_source = fallback_forecast(
                store=store,
                dept=dept,
                horizon=horizon,
                fallback_stats=fallback_stats,
            )
            method = "fallback"
            final_reason = "arima_failed"
    else:
        forecast, fallback_source = fallback_forecast(
            store=store,
            dept=dept,
            horizon=horizon,
            fallback_stats=fallback_stats,
        )
        method = "fallback"
        final_reason = reason

    pred_df = pd.DataFrame({
        "Store": store,
        "Dept": dept,
        "Date": future_dates,
        "Weekly_Sales_pred": forecast,
    })

    metadata = {
        "Store": store,
        "Dept": dept,
        "method": method,
        "reason": final_reason,
        "fallback_source": fallback_source,
        "history_rows": len(pair_history),
        "last_history_date": pair_history["Date"].max() if len(pair_history) > 0 else pd.NaT,
    }

    return pred_df, metadata

In [33]:
import time


def forecast_all_series_arima(
    history_df: pd.DataFrame,
    future_df: pd.DataFrame,
    order: tuple[int, int, int] = PRIMARY_ARIMA_ORDER,
    fallback_stats: dict | None = None,
    min_observations: int = MIN_OBSERVATIONS_FOR_ARIMA,
    max_weeks_since_last_observation: int = 13,
    progress_every: int | None = 250,
) -> dict:
    """
    Fit/forecast ARIMA separately for every Store-Dept pair in future_df.

    The function forecasts the full future date horizon for each pair, then
    merges predictions back to the exact Store-Dept-Date rows requested in
    future_df.

    This allows ARIMA to score all raw validation rows, including pairs that
    do not have a complete 39-week validation horizon.
    """
    t0 = time.perf_counter()

    history_df = history_df.copy()
    future_df = future_df.copy()

    history_df["Date"] = pd.to_datetime(history_df["Date"])
    future_df["Date"] = pd.to_datetime(future_df["Date"])

    key_cols = ["Store", "Dept", "Date"]

    assert not future_df.duplicated(key_cols).any(), "future_df has duplicate Store-Dept-Date rows"

    if fallback_stats is None:
        fallback_stats = build_fallback_stats(history_df, lookback_weeks=13)

    future_dates = pd.Index(sorted(future_df["Date"].unique()))
    split_end_date = history_df["Date"].max()

    future_pairs = (
        future_df[["Store", "Dept"]]
        .drop_duplicates()
        .sort_values(["Store", "Dept"])
        .reset_index(drop=True)
    )

    all_full_predictions = []
    metadata_rows = []

    for idx, row in future_pairs.iterrows():
        store = int(row["Store"])
        dept = int(row["Dept"])

        pair_pred_df, metadata = forecast_one_pair_arima(
            history_df=history_df,
            future_dates=future_dates,
            store=store,
            dept=dept,
            order=order,
            fallback_stats=fallback_stats,
            split_end_date=split_end_date,
            min_observations=min_observations,
            max_weeks_since_last_observation=max_weeks_since_last_observation,
        )

        all_full_predictions.append(pair_pred_df)
        metadata_rows.append(metadata)

        if progress_every is not None and (idx + 1) % progress_every == 0:
            print(f"Processed {idx + 1}/{len(future_pairs)} Store-Dept pairs")

    full_pred_df = pd.concat(all_full_predictions, ignore_index=True)
    metadata_df = pd.DataFrame(metadata_rows)

    assert not full_pred_df.duplicated(key_cols).any(), "full_pred_df has duplicate Store-Dept-Date rows"

    requested_keys = future_df[key_cols].copy()

    pred_rows = requested_keys.merge(
        full_pred_df,
        on=key_cols,
        how="left",
    )

    assert len(pred_rows) == len(future_df)
    assert pred_rows["Weekly_Sales_pred"].notna().all()

    elapsed_seconds = time.perf_counter() - t0

    stats = {
        "total_pairs": int(len(future_pairs)),
        "arima_pairs": int((metadata_df["method"] == "arima").sum()),
        "fallback_pairs": int((metadata_df["method"] == "fallback").sum()),
        "future_rows": int(len(future_df)),
        "future_horizon_weeks": int(len(future_dates)),
        "elapsed_seconds": float(elapsed_seconds),
        "seconds_per_pair": float(elapsed_seconds / max(len(future_pairs), 1)),
    }

    reason_counts = metadata_df["reason"].value_counts().to_dict()
    fallback_source_counts = (
        metadata_df["fallback_source"]
        .fillna("none")
        .value_counts()
        .to_dict()
    )

    return {
        "pred_rows": pred_rows,
        "full_pred_df": full_pred_df,
        "metadata_df": metadata_df,
        "stats": stats,
        "reason_counts": reason_counts,
        "fallback_source_counts": fallback_source_counts,
    }

In [35]:
def log_arima_validation_run(
    run_name: str,
    validation_strategy: str,
    arima_order: tuple[int, int, int],
    arima_result: dict,
    arima_eval: dict,
    history_df: pd.DataFrame,
    valid_df: pd.DataFrame,
):
    mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_param("model_family", "ARIMA")
        mlflow.log_param("order", str(arima_order))
        mlflow.log_param("validation_strategy", validation_strategy)
        mlflow.log_param("prediction_length", PREDICTION_LENGTH)
        mlflow.log_param("min_observations_for_arima", MIN_OBSERVATIONS_FOR_ARIMA)
        mlflow.log_param("max_weeks_since_last_observation", 13)
        mlflow.log_param("fallback_method", FALLBACK_METHOD)
        mlflow.log_param("fallback_strategy", "hierarchical_recent_mean")
        mlflow.log_param("missing_weeks_strategy", "observed_points_only_no_zero_fill")
        mlflow.log_param("preprocessing_run_id", arima_preprocessing_run_id)

        mlflow.log_metric("history_rows", len(history_df))
        mlflow.log_metric("valid_rows", len(valid_df))
        mlflow.log_metric("valid_wmae", arima_eval["valid_wmae"])
        mlflow.log_metric("valid_mae", arima_eval["valid_mae"])

        for key, value in arima_result["stats"].items():
            mlflow.log_metric(key, value)

        for key, value in arima_result["reason_counts"].items():
            mlflow.log_metric(f"reason_{key}", value)

        for key, value in arima_result["fallback_source_counts"].items():
            mlflow.log_metric(f"fallback_source_{key}", value)

        artifact_dir = repo_root / "artifacts" / "arima"
        artifact_dir.mkdir(parents=True, exist_ok=True)

        pred_path = artifact_dir / f"{validation_strategy}_predictions.csv"
        metadata_path = artifact_dir / f"{validation_strategy}_metadata.csv"

        arima_result["pred_rows"].to_csv(pred_path, index=False)
        arima_result["metadata_df"].to_csv(metadata_path, index=False)

        mlflow.log_artifact(str(pred_path), artifact_path="predictions")
        mlflow.log_artifact(str(metadata_path), artifact_path="metadata")

        run_id = run.info.run_id

    print(f"Logged {run_name}:", run_id)
    return run_id

## ARIMA Experiments: Last-39 Validation 

In [38]:
sample_pairs = (
    last39_valid_raw[["Store", "Dept"]]
    .drop_duplicates()
    .sort_values(["Store", "Dept"])
    .head(20)
)

last39_valid_sample = last39_valid_raw.merge(
    sample_pairs,
    on=["Store", "Dept"],
    how="inner",
)

print("Sample validation rows:", len(last39_valid_sample))
print("Sample validation pairs:", last39_valid_sample[["Store", "Dept"]].drop_duplicates().shape[0])

sample_arima_result = forecast_all_series_arima(
    history_df=last39_train_raw,
    future_df=last39_valid_sample,
    order=PRIMARY_ARIMA_ORDER,
    fallback_stats=last39_fallback_stats,
    min_observations=MIN_OBSERVATIONS_FOR_ARIMA,
    max_weeks_since_last_observation=13,
    progress_every=5,
)

sample_arima_eval = evaluate_arima_validation(
    valid_df=last39_valid_sample,
    pred_rows=sample_arima_result["pred_rows"],
)

print("Sample ARIMA WMAE:", sample_arima_eval["valid_wmae"])
print("Sample ARIMA MAE:", sample_arima_eval["valid_mae"])
print("Stats:", sample_arima_result["stats"])
print("Reason counts:", sample_arima_result["reason_counts"])
print("Fallback source counts:", sample_arima_result["fallback_source_counts"])

display(sample_arima_result["metadata_df"].head())
display(sample_arima_eval["eval_df"].head())

Sample validation rows: 771
Sample validation pairs: 20
Processed 5/20 Store-Dept pairs
Processed 10/20 Store-Dept pairs
Processed 15/20 Store-Dept pairs
Processed 20/20 Store-Dept pairs
Sample ARIMA WMAE: 4771.376087837014
Sample ARIMA MAE: 4734.100288633034
Stats: {'total_pairs': 20, 'arima_pairs': 20, 'fallback_pairs': 0, 'future_rows': 771, 'future_horizon_weeks': 39}
Reason counts: {'arima_success': 20}
Fallback source counts: {'none': 20}


,Store,Dept,method,reason,fallback_source,history_rows,last_history_date
0,1,1,arima,arima_success,None,104,2012-01-27
1,1,2,arima,arima_success,None,104,2012-01-27
2,1,3,arima,arima_success,None,104,2012-01-27
3,1,4,arima,arima_success,None,104,2012-01-27
4,1,5,arima,arima_success,None,104,2012-01-27


,Store,Dept,Date,Weekly_Sales,IsHoliday,Weekly_Sales_pred
0,1,1,2012-02-03,23510.49,False,20896.492288
1,1,1,2012-02-10,36988.49,True,22108.877304
2,1,1,2012-02-17,54060.10,False,22692.548266
3,1,1,2012-02-24,20124.22,False,22973.541339
4,1,1,2012-03-02,20113.03,False,23108.818083


In [39]:
last39_arima_result = forecast_all_series_arima(
    history_df=last39_train_raw,
    future_df=last39_valid_raw,
    order=PRIMARY_ARIMA_ORDER,
    fallback_stats=last39_fallback_stats,
    min_observations=MIN_OBSERVATIONS_FOR_ARIMA,
    max_weeks_since_last_observation=13,
    progress_every=250,
)

last39_arima_eval = evaluate_arima_validation(
    valid_df=last39_valid_raw,
    pred_rows=last39_arima_result["pred_rows"],
)

print("Last-39 ARIMA validation")
print("WMAE:", last39_arima_eval["valid_wmae"])
print("MAE:", last39_arima_eval["valid_mae"])

print("\nStats:")
print(last39_arima_result["stats"])

print("\nReason counts:")
print(last39_arima_result["reason_counts"])

print("\nFallback source counts:")
print(last39_arima_result["fallback_source_counts"])

display(last39_arima_result["metadata_df"].head())
display(last39_arima_eval["eval_df"].head())

Processed 250/3204 Store-Dept pairs
Processed 500/3204 Store-Dept pairs
Processed 750/3204 Store-Dept pairs
Processed 1000/3204 Store-Dept pairs
Processed 1250/3204 Store-Dept pairs
Processed 1500/3204 Store-Dept pairs
Processed 1750/3204 Store-Dept pairs
Processed 2000/3204 Store-Dept pairs
Processed 2250/3204 Store-Dept pairs
Processed 2500/3204 Store-Dept pairs
Processed 2750/3204 Store-Dept pairs
Processed 3000/3204 Store-Dept pairs
Last-39 ARIMA validation
WMAE: 2743.4561037452236
MAE: 2743.505417758898

Stats:
{'total_pairs': 3204, 'arima_pairs': 2981, 'fallback_pairs': 223, 'future_rows': 115588, 'future_horizon_weeks': 39}

Reason counts:
{'arima_success': 2981, 'too_few_observations': 190, 'no_pair_history': 25, 'stale_series': 8}

Fallback source counts:
{'none': 2981, 'pair_recent_mean': 140, 'dept_recent_mean': 83}


,Store,Dept,method,reason,fallback_source,history_rows,last_history_date
0,1,1,arima,arima_success,None,104,2012-01-27
1,1,2,arima,arima_success,None,104,2012-01-27
2,1,3,arima,arima_success,None,104,2012-01-27
3,1,4,arima,arima_success,None,104,2012-01-27
4,1,5,arima,arima_success,None,104,2012-01-27


,Store,Dept,Date,Weekly_Sales,IsHoliday,Weekly_Sales_pred
0,1,1,2012-02-03,23510.49,False,20896.492288
1,1,1,2012-02-10,36988.49,True,22108.877304
2,1,1,2012-02-17,54060.10,False,22692.548266
3,1,1,2012-02-24,20124.22,False,22973.541339
4,1,1,2012-03-02,20113.03,False,23108.818083


In [ ]:
arima_last39_run_id = log_arima_validation_run(
    run_name="ARIMA_Last39_order_1_1_1",
    validation_strategy="last39",
    arima_order=PRIMARY_ARIMA_ORDER,
    arima_result=last39_arima_result,
    arima_eval=last39_arima_eval,
    history_df=last39_train_raw,
    valid_df=last39_valid_raw,
)

In [44]:
KAGGLE_ARIMA_RUN_CONFIGS = [
    {
        "run_name": "ARIMA_Last39_order_0_1_0",
        "validation_strategy": "last39",
        "order": (0, 1, 0),
        "description": "Random walk baseline",
    },
    {
        "run_name": "ARIMA_Last39_order_1_0_0",
        "validation_strategy": "last39",
        "order": (1, 0, 0),
        "description": "AR only, no differencing",
    },
    # {
    #     "run_name": "ARIMA_Last39_order_1_1_1",
    #     "validation_strategy": "last39",
    #     "order": (1, 1, 1),
    #     "description": "Primary ARIMA model",
    # },
]

LOCAL_ARIMA_RUN_CONFIGS = [
    {
        "run_name": "ARIMA_Last39_order_1_1_0",
        "validation_strategy": "last39",
        "order": (1, 1, 0),
        "description": "Differenced AR model",
    },
    {
        "run_name": "ARIMA_Last39_order_0_1_1",
        "validation_strategy": "last39",
        "order": (0, 1, 1),
        "description": "Differenced MA model",
    },
]

In [45]:
def run_last39_arima_config(config):
    print("=" * 80)
    print(config["run_name"])
    print("Order:", config["order"])
    print(config.get("description", ""))

    result = forecast_all_series_arima(
        history_df=last39_train_raw,
        future_df=last39_valid_raw,
        order=config["order"],
        fallback_stats=last39_fallback_stats,
        min_observations=MIN_OBSERVATIONS_FOR_ARIMA,
        max_weeks_since_last_observation=13,
        progress_every=250,
    )

    evaluation = evaluate_arima_validation(
        valid_df=last39_valid_raw,
        pred_rows=result["pred_rows"],
    )

    print("WMAE:", evaluation["valid_wmae"])
    print("MAE:", evaluation["valid_mae"])
    print("Stats:", result["stats"])
    print("Reason counts:", result["reason_counts"])
    print("Fallback source counts:", result["fallback_source_counts"])

    run_id = log_arima_validation_run(
        run_name=config["run_name"],
        validation_strategy=config["validation_strategy"],
        arima_order=config["order"],
        arima_result=result,
        arima_eval=evaluation,
        history_df=last39_train_raw,
        valid_df=last39_valid_raw,
    )

    return {
        "run_name": config["run_name"],
        "order": config["order"],
        "valid_wmae": evaluation["valid_wmae"],
        "valid_mae": evaluation["valid_mae"],
        "run_id": run_id,
        "result": result,
        "evaluation": evaluation,
    }

In [46]:
arima_run_results = []

for config in KAGGLE_ARIMA_RUN_CONFIGS:  # LOCAL_ARIMA_RUN_CONFIGS
    arima_run_results.append(run_last39_arima_config(config))

summary_df = pd.DataFrame([
    {
        "run_name": r["run_name"],
        "order": str(r["order"]),
        "valid_wmae": r["valid_wmae"],
        "valid_mae": r["valid_mae"],
        "run_id": r["run_id"],
    }
    for r in arima_run_results
]).sort_values("valid_wmae")

display(summary_df)

ARIMA_Last39_order_0_1_0
Order: (0, 1, 0)
Random walk baseline
Processed 250/3204 Store-Dept pairs
Processed 500/3204 Store-Dept pairs
Processed 750/3204 Store-Dept pairs
Processed 1000/3204 Store-Dept pairs
Processed 1250/3204 Store-Dept pairs
Processed 1500/3204 Store-Dept pairs
Processed 1750/3204 Store-Dept pairs
Processed 2000/3204 Store-Dept pairs
Processed 2250/3204 Store-Dept pairs
Processed 2500/3204 Store-Dept pairs
Processed 2750/3204 Store-Dept pairs
Processed 3000/3204 Store-Dept pairs
WMAE: 3225.113299140443
MAE: 3155.0822827133784
Stats: {'total_pairs': 3204, 'arima_pairs': 2981, 'fallback_pairs': 223, 'future_rows': 115588, 'future_horizon_weeks': 39}
Reason counts: {'arima_success': 2981, 'too_few_observations': 190, 'no_pair_history': 25, 'stale_series': 8}
Fallback source counts: {'none': 2981, 'pair_recent_mean': 140, 'dept_recent_mean': 83}
🏃 View run ARIMA_Last39_order_0_1_0 at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/

,run_name,order,valid_wmae,valid_mae,run_id
1,ARIMA_Last39_order_1_0_0,"(1, 0, 0)",2588.448243,2562.468790,0440e65223d84f35b90eb7a527373fd6
0,ARIMA_Last39_order_0_1_0,"(0, 1, 0)",3225.113299,3155.082283,b8ce7e4bd00b426299c0533da8e87ee3


## ARIMA Experiments: Calendar-aligned 

In [53]:
KAGGLE_CALENDAR_ARIMA_RUN_CONFIGS = [
    {
        "run_name": "ARIMA_Calendar_order_0_1_0",
        "validation_strategy": "calendar",
        "order": (0, 1, 0),
        "description": "Random walk baseline calendar check",
    },
    {
        "run_name": "ARIMA_Calendar_order_1_1_1",
        "validation_strategy": "calendar",
        "order": (1, 1, 1),
        "description": "Primary ARIMA model calendar check",
    },
]

In [54]:
LOCAL_CALENDAR_ARIMA_RUN_CONFIGS = [
    {
        "run_name": "ARIMA_Calendar_order_1_1_0",
        "validation_strategy": "calendar",
        "order": (1, 1, 0),
        "description": "Differenced AR model calendar check",
    },
    {
        "run_name": "ARIMA_Calendar_order_0_1_1",
        "validation_strategy": "calendar",
        "order": (0, 1, 1),
        "description": "Differenced MA model calendar check",
    },
    {
        "run_name": "ARIMA_Calendar_order_1_0_0",
        "validation_strategy": "calendar",
        "order": (1, 0, 0),
        "description": "AR-only model without differencing calendar check",
    },
]

In [55]:
def run_calendar_arima_config(config):
    print("=" * 80)
    print(config["run_name"])
    print("Order:", config["order"])
    print(config.get("description", ""))

    result = forecast_all_series_arima(
        history_df=calendar_train_raw,
        future_df=calendar_valid_raw,
        order=config["order"],
        fallback_stats=calendar_fallback_stats,
        min_observations=MIN_OBSERVATIONS_FOR_ARIMA,
        max_weeks_since_last_observation=13,
        progress_every=250,
    )

    evaluation = evaluate_arima_validation(
        valid_df=calendar_valid_raw,
        pred_rows=result["pred_rows"],
    )

    print("Calendar WMAE:", evaluation["valid_wmae"])
    print("Calendar MAE:", evaluation["valid_mae"])
    print("Stats:", result["stats"])
    print("Reason counts:", result["reason_counts"])
    print("Fallback source counts:", result["fallback_source_counts"])

    run_id = log_arima_validation_run(
        run_name=config["run_name"],
        validation_strategy="calendar",
        arima_order=config["order"],
        arima_result=result,
        arima_eval=evaluation,
        history_df=calendar_train_raw,
        valid_df=calendar_valid_raw,
    )

    return {
        "run_name": config["run_name"],
        "order": config["order"],
        "valid_wmae": evaluation["valid_wmae"],
        "valid_mae": evaluation["valid_mae"],
        "run_id": run_id,
        "result": result,
        "evaluation": evaluation,
    }

In [56]:
local_calendar_arima_run_results = []

for config in KAGGLE_CALENDAR_ARIMA_RUN_CONFIGS: # LOCAL_CALENDAR_ARIMA_RUN_CONFIGS
    local_calendar_arima_run_results.append(run_calendar_arima_config(config))

local_calendar_summary_df = pd.DataFrame([
    {
        "run_name": r["run_name"],
        "order": str(r["order"]),
        "calendar_wmae": r["valid_wmae"],
        "calendar_mae": r["valid_mae"],
        "run_id": r["run_id"],
    }
    for r in local_calendar_arima_run_results
]).sort_values("calendar_wmae")

display(local_calendar_summary_df)

ARIMA_Calendar_order_0_1_0
Order: (0, 1, 0)
Random walk baseline calendar check
Processed 250/3233 Store-Dept pairs
Processed 500/3233 Store-Dept pairs
Processed 750/3233 Store-Dept pairs
Processed 1000/3233 Store-Dept pairs
Processed 1250/3233 Store-Dept pairs
Processed 1500/3233 Store-Dept pairs
Processed 1750/3233 Store-Dept pairs
Processed 2000/3233 Store-Dept pairs
Processed 2250/3233 Store-Dept pairs
Processed 2500/3233 Store-Dept pairs
Processed 2750/3233 Store-Dept pairs
Processed 3000/3233 Store-Dept pairs
Calendar WMAE: 3886.4399679566886
Calendar MAE: 3539.390537850912
Stats: {'total_pairs': 3233, 'arima_pairs': 2969, 'fallback_pairs': 264, 'future_rows': 115856, 'future_horizon_weeks': 39}
Reason counts: {'arima_success': 2969, 'too_few_observations': 192, 'no_pair_history': 68, 'stale_series': 4}
Fallback source counts: {'none': 2969, 'pair_recent_mean': 140, 'dept_recent_mean': 121, 'store_recent_mean': 3}
🏃 View run ARIMA_Calendar_order_0_1_0 at: https://dagshub.com/Luka

,run_name,order,calendar_wmae,calendar_mae,run_id
1,ARIMA_Calendar_order_1_1_1,"(1, 1, 1)",3550.717330,3145.098503,8c6a1b99aaa54f46a1ead27273ae2107
0,ARIMA_Calendar_order_0_1_0,"(0, 1, 0)",3886.439968,3539.390538,0faaa1384afc43d69427b23d35d8c993


## Kaggle Check


In [59]:
# Full-train ARIMA setup for final test inference

full_train_arima = train.copy()
full_test_arima = test.copy()

full_train_arima["Date"] = pd.to_datetime(full_train_arima["Date"])
full_test_arima["Date"] = pd.to_datetime(full_test_arima["Date"])

full_train_arima = full_train_arima.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
full_test_arima = full_test_arima.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)

final_fallback_stats = build_fallback_stats(
    full_train_arima,
    lookback_weeks=13,
)

print("Full train rows:", len(full_train_arima))
print("Full test rows:", len(full_test_arima))
print("Full train date range:", full_train_arima["Date"].min(), "->", full_train_arima["Date"].max())
print("Full test date range:", full_test_arima["Date"].min(), "->", full_test_arima["Date"].max())
print("Full train Store-Dept pairs:", full_train_arima[["Store", "Dept"]].drop_duplicates().shape[0])
print("Full test Store-Dept pairs:", full_test_arima[["Store", "Dept"]].drop_duplicates().shape[0])
print("Final global fallback mean:", final_fallback_stats["global_recent_mean"])

Full train rows: 421570
Full test rows: 115064
Full train date range: 2010-02-05 00:00:00 -> 2012-10-26 00:00:00
Full test date range: 2012-11-02 00:00:00 -> 2013-07-26 00:00:00
Full train Store-Dept pairs: 3331
Full test Store-Dept pairs: 3169
Final global fallback mean: 15620.503259018944


In [60]:
test_pair_coverage = summarize_pair_overlap(
    full_train_arima,
    full_test_arima,
    "Final test pair coverage",
)

print("Final test unique dates:", full_test_arima["Date"].nunique())
print("Expected prediction length:", PREDICTION_LENGTH)

assert full_test_arima["Date"].nunique() == PREDICTION_LENGTH
assert full_test_arima["IsHoliday"].notna().all()

Final test pair coverage
History pairs: 3331
Future/validation pairs: 3169
Overlapping pairs: 3158
Unseen future pairs: 11
Future/validation rows: 115064

Unseen pairs:


,Store,Dept
360,5,99
648,9,99
723,10,99
1275,18,43
1714,24,43
1821,25,99
2413,34,39
2545,36,30
2605,37,29
2948,42,30


Final test unique dates: 39
Expected prediction length: 39


In [61]:
BEST_ARIMA_FINAL_ORDER = (1, 0, 0)

ARIMA_FINAL_CONFIG = {
    "model_family": "ARIMA",
    "order": BEST_ARIMA_FINAL_ORDER,
    "fallback_strategy": "hierarchical_recent_mean",
    "fallback_lookback_weeks": 13,
    "min_observations_for_arima": MIN_OBSERVATIONS_FOR_ARIMA,
    "max_weeks_since_last_observation": 13,
    "missing_weeks_strategy": "observed_points_only_no_zero_fill",
}

print("Final ARIMA config:")
print(ARIMA_FINAL_CONFIG)

Final ARIMA config:
{'model_family': 'ARIMA', 'order': (1, 0, 0), 'fallback_strategy': 'hierarchical_recent_mean', 'fallback_lookback_weeks': 13, 'min_observations_for_arima': 30, 'max_weeks_since_last_observation': 13, 'missing_weeks_strategy': 'observed_points_only_no_zero_fill'}


In [62]:
final_arima_result = forecast_all_series_arima(
    history_df=full_train_arima,
    future_df=full_test_arima,
    order=BEST_ARIMA_FINAL_ORDER,
    fallback_stats=final_fallback_stats,
    min_observations=MIN_OBSERVATIONS_FOR_ARIMA,
    max_weeks_since_last_observation=13,
    progress_every=250,
)

print("Final ARIMA test prediction stats:")
print(final_arima_result["stats"])

print("\nReason counts:")
print(final_arima_result["reason_counts"])

print("\nFallback source counts:")
print(final_arima_result["fallback_source_counts"])

display(final_arima_result["metadata_df"].head())
display(final_arima_result["pred_rows"].head())

assert len(final_arima_result["pred_rows"]) == len(full_test_arima)
assert final_arima_result["pred_rows"]["Weekly_Sales_pred"].notna().all()

Processed 250/3169 Store-Dept pairs
Processed 500/3169 Store-Dept pairs
Processed 750/3169 Store-Dept pairs
Processed 1000/3169 Store-Dept pairs
Processed 1250/3169 Store-Dept pairs
Processed 1500/3169 Store-Dept pairs
Processed 1750/3169 Store-Dept pairs
Processed 2000/3169 Store-Dept pairs
Processed 2250/3169 Store-Dept pairs
Processed 2500/3169 Store-Dept pairs
Processed 2750/3169 Store-Dept pairs
Processed 3000/3169 Store-Dept pairs
Final ARIMA test prediction stats:
{'total_pairs': 3169, 'arima_pairs': 3014, 'fallback_pairs': 155, 'future_rows': 115064, 'future_horizon_weeks': 39}

Reason counts:
{'arima_success': 3014, 'too_few_observations': 129, 'stale_series': 15, 'no_pair_history': 11}

Fallback source counts:
{'none': 3014, 'dept_recent_mean': 82, 'pair_recent_mean': 70, 'store_recent_mean': 3}


,Store,Dept,method,reason,fallback_source,history_rows,last_history_date
0,1,1,arima,arima_success,None,143,2012-10-26
1,1,2,arima,arima_success,None,143,2012-10-26
2,1,3,arima,arima_success,None,143,2012-10-26
3,1,4,arima,arima_success,None,143,2012-10-26
4,1,5,arima,arima_success,None,143,2012-10-26


,Store,Dept,Date,Weekly_Sales_pred
0,1,1,2012-11-02,25255.605197
1,1,1,2012-11-09,24055.123438
2,1,1,2012-11-16,23380.173450
3,1,1,2012-11-23,23000.694560
4,1,1,2012-11-30,22787.339154


In [63]:
arima_submission_df = final_arima_result["pred_rows"].copy()

arima_submission_df["Weekly_Sales"] = arima_submission_df["Weekly_Sales_pred"].clip(lower=0)

arima_submission_df["Id"] = (
    arima_submission_df["Store"].astype(str)
    + "_"
    + arima_submission_df["Dept"].astype(str)
    + "_"
    + pd.to_datetime(arima_submission_df["Date"]).dt.strftime("%Y-%m-%d")
)

arima_submission_df = arima_submission_df[["Id", "Weekly_Sales"]]

assert len(arima_submission_df) == len(test)
assert arima_submission_df["Weekly_Sales"].notna().all()

print(arima_submission_df.head())
print(arima_submission_df.shape)
print(arima_submission_df["Weekly_Sales"].describe())

order_str = "_".join(map(str, BEST_ARIMA_FINAL_ORDER))

submission_name = f"submission_arima_order_{order_str}.csv"
submission_path = Path("/kaggle/working") / submission_name

arima_submission_df.to_csv(submission_path, index=False)

print("Saved submission to:", submission_path)

               Id  Weekly_Sales
0  1_1_2012-11-02  25255.605197
1  1_1_2012-11-09  24055.123438
2  1_1_2012-11-16  23380.173450
3  1_1_2012-11-23  23000.694560
4  1_1_2012-11-30  22787.339154
(115064, 2)
count    115064.000000
mean      15932.758378
std       21559.637443
min           0.000000
25%        2383.939154
50%        7951.060899
75%       20145.902706
max      182527.956014
Name: Weekly_Sales, dtype: float64
Saved submission to: /kaggle/working/submission_arima_order_1_0_0.csv


`ARIMA_order_1_0_0` clearly outperformed other orders, but the overall test result is significantly worse than other models that we experimented on, which was not an unexpected outcome.